# Example GitHub Query and Classification

This notebook runs a small GitHub Code Search query and classifies example code with `vi_github_classifier.classify_source`, which applies the same extraction and classification rules as `github-advi-usage analyze`. The live query cell requires `GITHUB_TOKEN`; the classification cells work offline.

In [1]:
import json
import os
import sys
import urllib.parse
import urllib.request
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from github_advi_usage.vi_github_classifier import classify_source

In [2]:
query = '"pm.fit" "method" "advi" language:Python'
token = os.environ.get('GITHUB_TOKEN')
if token:
    url = 'https://api.github.com/search/code?' + urllib.parse.urlencode({'q': query, 'per_page': 1})
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'})
    result = json.loads(urllib.request.urlopen(req).read().decode('utf-8'))
    print(json.dumps(result.get('items', [])[:1], indent=2)[:2000])
else:
    print('Set GITHUB_TOKEN to run a live GitHub query. Using the offline example below.')


Set GITHUB_TOKEN to run a live GitHub query. Using the offline example below.


In [3]:
source = '''
import pymc as pm

with model:
    approx = pm.fit(method="advi")
    approx_mc = pm.fit(method="advi", obj_n_mc=10)
'''
result = classify_source(source, repo_full_name="example/repo", path="model.py")
[(row["line_number"], row["final_classification"], row["classification_reason"]) for row in result["call_site_evidence"]]

[(6, 'adjusted_mc', 'Adjusted MC/sample-count or ELBO particle behavior: 10.'),
 (5,
  'default',
  'True-positive VI fit call with default relevant optimizer and MC/sample-count behavior.')]